# Evaluating Performance of Linear Chain CRF on the CoNLL-2003 Dataset

## Load Dependencies

In [1]:
from collections import Counter
from datasets import concatenate_datasets, DatasetDict, load_dataset
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from torchcrf import CRF

## Set Global Parameters

In [2]:
BATCH_N = 32
EMBED_DIM = 300
LEARNING_RATE = 1e-3
NUM_EPOCHS = 30

## Load Data

In [3]:
# Data downloaded from https://nlp.stanford.edu/projects/glove/
glove_path = "data/dolma_300_2024_1.2M.100_combined.txt"

embeddings_dict = {}

with open(file = glove_path, mode = 'r', encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], "float32")
        embeddings_dict[word] = vector

In [4]:
BASE = "https://huggingface.co/datasets/conll2003/resolve/refs%2Fconvert%2Fparquet/conll2003"

dataset = DatasetDict({
    split: load_dataset("parquet", data_files={split: f"{BASE}/{split}/0000.parquet"}, split=split)
    for split in ["train", "validation", "test"]
})

NUM_POS_CLASSES = dataset['train'].features['pos_tags'].feature.num_classes
NUM_CHUNK_CLASSES = dataset['train'].features['chunk_tags'].feature.num_classes

## Define Model and Utilities

In [6]:
def get_affixes(corpus, kind, affix_length):

    affs = Counter()
    for seq in corpus:
        eligible_toks = [t.lower() for t in seq if len(t) > 2]

        for t in eligible_toks:
            index = min(affix_length, len(t)-1)
            for i in range(index):
                if kind == "prefix":
                    val = t[:(index-i)]
                elif kind == "suffix":
                    val = t[-1*(index-i):]
                else:
                    raise ValueError("Only prefix and suffix are supported.")
                affs[val] += 1
    return affs

def get_top_affixes(corpus, top=100, affix_length=4):
    all_pre = get_affixes(corpus, kind="prefix", affix_length=affix_length)
    top_pre = all_pre.most_common(top)
    all_suf = get_affixes(corpus, kind="suffix", affix_length=affix_length)
    top_suf = all_suf.most_common(top)

    return top_pre, top_suf

In [7]:
sent_1 = ["We", "synergize", "at", "work"]
sent_2 = ["I", "require", "the", "next", "size"]
sent_3 = ["I", "will", "synthesize", "it"]

corpus = [sent_1, sent_2, sent_3]

pres, sufs = get_top_affixes(corpus, top=3)

print("Prefix:", pres)
print("Suffix:", sufs)

sent_test = ["We", "realize", "it", "is", "synthetic"]

print("\n---Prefix Test---")
for t in sent_test:
    for p in pres:
        if p[0] in t:
            print(p[0])
print("\n---Suffix Test---")
for t in sent_test:
    for s in sufs:
        if s[0] in t:
            print(s[0])

Prefix: [('s', 3), ('syn', 2), ('sy', 2)]
Suffix: [('e', 5), ('ize', 3), ('ze', 3)]

---Prefix Test---
s
s
syn
sy

---Suffix Test---
e
e
ize
ze
e


In [8]:
OOV_VECTOR = np.zeros(EMBED_DIM, dtype="float32")

class NERDataset(Dataset):
    '''
    Prepares dataset for Pytorch's `DataLoader`.
    '''
    def __init__(self, tokens, tags, pos_tags, c_tags, embed_map, affix_size=100):
        self.samples = [(t, g, p, c) for t, g, p, c in zip(tokens, tags, pos_tags, c_tags) if len(t)>0]
        self.embed_map = embed_map
        top_prefix, top_suffix = get_top_affixes(tokens, top=affix_size)
        self.prefix_vocab = {aff:i+1 for i, (aff, _) in enumerate(top_prefix)}
        self.suffix_vocab = {aff:i+1 for i, (aff, _) in enumerate(top_suffix)}
        self.affix_size = affix_size + 1

    def get_multihot(self, tok, kind):
        vec = torch.zeros(self.affix_size)
        for n in range(1, 5):
            if kind == "prefix":
                aff = tok.lower()[:n] if len(tok) >= n else None
                if aff in self.prefix_vocab:
                    vec[self.prefix_vocab[aff]] = 1.0
            elif kind == "suffix":
                aff = tok.lower()[-n:] if len(tok) >= n else None
                if aff in self.suffix_vocab:
                    vec[self.suffix_vocab[aff]] = 1.0
            else:
                raise ValueError("Only prefix and suffix are supported.")
        return vec

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        toks, tags, pos, chnk = self.samples[idx]
        vecs = torch.stack([
            torch.cat([
                torch.from_numpy(self.embed_map.get(t, self.embed_map.get(t.lower(), OOV_VECTOR))),
                torch.nn.functional.one_hot(torch.tensor(p), num_classes=NUM_POS_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(pos[max(i-1, 0)]), num_classes=NUM_POS_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(pos[min(i+1, len(toks)-1)]), num_classes=NUM_POS_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(c), num_classes=NUM_CHUNK_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(chnk[max(i-1, 0)]), num_classes=NUM_CHUNK_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(chnk[min(i+1, len(toks)-1)]), num_classes=NUM_CHUNK_CLASSES).float(),
                torch.nn.functional.one_hot(torch.tensor(int(t.isdigit())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[max(i-1, 0)].isdigit())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[min(i+1, len(toks)-1)].isdigit())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(any([i.isdigit() for i in t]))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(any([i.isdigit() for i in toks[max(i-1, 0)]]))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(any([i.isdigit() for i in toks[min(i+1, len(toks)-1)]]))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(t.isupper())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[max(i-1, 0)].isupper())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[min(i+1, len(toks)-1)].isupper())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(t.istitle())), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(t.endswith(('ing', 'ed', 'ly', 'ion', 'tion', 'ity')))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[max(i-1, 0)].endswith(('ing', 'ed', 'ly', 'ion', 'tion', 'ity')))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(toks[min(i+1, len(toks)-1)].endswith(('ing', 'ed', 'ly', 'ion', 'tion', 'ity')))), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int('-' in t)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int('-' in toks[max(i-1, 0)])), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int('-' in toks[min(i+1, len(toks)-1)])), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(i==0)), num_classes=2),
                torch.nn.functional.one_hot(torch.tensor(int(i==len(toks)-1)), num_classes=2),
                self.get_multihot(t, kind="suffix"),
                self.get_multihot(t, kind="prefix")
            ])
            for i, (t, p, c) in enumerate(zip(toks, pos, chnk))
        ])
        return vecs, torch.tensor(tags, dtype=torch.long)


def collate_fn(batch):
    '''
    Defines padding and masking for Pytorch's `DataLoader`.
    Returns a tuple.
    '''
    seqs, tags = zip(*batch)
    X = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    y = pad_sequence(tags, batch_first=True, padding_value=-1)
    mask = (y != -1).bool()
    return X, y, mask

In [9]:
class LinearChainCRF(nn.Module):
    ''' 
    A simple linear chain CRF for NLP tasks.
    First flattens embeddings to match number of NER tags,
    then passes results to the CRF.
    '''
    def __init__(self, embed_dim, num_tags):
        super().__init__()
        self.linear = nn.Linear(embed_dim, num_tags)
        self.crf = CRF(num_tags=num_tags, batch_first=True)

    def forward(self, x, tags, mask):
        emissions = self.linear(x)
        return -self.crf(emissions, tags, mask=mask)

    def decode(self, x, mask):
        emissions = self.linear(x)
        return self.crf.decode(emissions, mask=mask)

## Training

In [ ]:
num_tags = dataset['train'].features['ner_tags'].feature.num_classes
model = LinearChainCRF(EMBED_DIM+3*NUM_POS_CLASSES+3*NUM_CHUNK_CLASSES+18*2+2*101, num_tags)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_loader = DataLoader(
    NERDataset(
        dataset["train"]["tokens"], 
        dataset["train"]["ner_tags"], 
        dataset['train']['pos_tags'], 
        dataset['train']['chunk_tags'], 
        embeddings_dict
    ),
    batch_size=BATCH_N, shuffle=True, collate_fn=collate_fn
)

model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    for X, y, mask in train_loader:
        optimizer.zero_grad()
        loss = model(X, y, mask) / mask.sum()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}  loss={total_loss:.4f}")


## Validation

In [ ]:
ner_labels = dataset['train'].features['ner_tags'].feature.names

val_loader = DataLoader(
    NERDataset(
        dataset["validation"]["tokens"], 
        dataset["validation"]["ner_tags"], 
        dataset['validation']['pos_tags'], 
        dataset['validation']['chunk_tags'], 
        embeddings_dict
    ),
    batch_size=BATCH_N, shuffle=False, collate_fn=collate_fn
)

model.eval()
all_preds, all_golds = [], []

with torch.no_grad():
    for X, y, mask in val_loader:
        batch_preds = model.decode(X, mask)
        for pred_seq, gold_seq, m in zip(batch_preds, y, mask):
            valid_len = m.sum().item()
            all_preds.append([ner_labels[i] for i in pred_seq[:valid_len]])
            all_golds.append([ner_labels[i] for i in gold_seq[:valid_len].tolist()])
            

In [ ]:
from seqeval.metrics import classification_report
from seqeval.metrics import f1_score

print(f"F1-score: {f1_score(all_golds, all_preds):.2f}")
print(classification_report(all_golds, all_preds))

## Results on Test Set

In [10]:
train_dataset = concatenate_datasets([dataset['train'], dataset['validation']])
num_tags = train_dataset.features['ner_tags'].feature.num_classes
model = LinearChainCRF(EMBED_DIM+3*NUM_POS_CLASSES+3*NUM_CHUNK_CLASSES+18*2+2*101, num_tags)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_loader = DataLoader(
    NERDataset(
        train_dataset["tokens"], 
        train_dataset["ner_tags"], 
        train_dataset['pos_tags'], 
        train_dataset['chunk_tags'], 
        embeddings_dict
    ),
    batch_size=BATCH_N, shuffle=True, collate_fn=collate_fn
)

model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    for X, y, mask in train_loader:
        optimizer.zero_grad()
        loss = model(X, y, mask) / mask.sum()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}  loss={total_loss:.4f}")

Epoch 1  loss=195.7707
Epoch 2  loss=85.0600
Epoch 3  loss=64.0755
Epoch 4  loss=53.6292
Epoch 5  loss=46.9330
Epoch 6  loss=42.5861
Epoch 7  loss=39.0009
Epoch 8  loss=36.6125
Epoch 9  loss=34.6552
Epoch 10  loss=33.1078
Epoch 11  loss=31.7667
Epoch 12  loss=30.5598
Epoch 13  loss=29.6957
Epoch 14  loss=28.9367
Epoch 15  loss=28.0763
Epoch 16  loss=27.6694
Epoch 17  loss=27.0768
Epoch 18  loss=26.5649
Epoch 19  loss=26.0861
Epoch 20  loss=25.7458
Epoch 21  loss=25.3752
Epoch 22  loss=25.1554
Epoch 23  loss=24.8250
Epoch 24  loss=24.5383
Epoch 25  loss=24.3401
Epoch 26  loss=24.1377
Epoch 27  loss=23.8724
Epoch 28  loss=23.8262
Epoch 29  loss=23.7664
Epoch 30  loss=23.5782


In [11]:
ner_labels = train_dataset.features['ner_tags'].feature.names

test_loader = DataLoader(
    NERDataset(
        dataset["test"]["tokens"], 
        dataset["test"]["ner_tags"], 
        dataset['test']['pos_tags'], 
        dataset['test']['chunk_tags'], 
        embeddings_dict
    ),
    batch_size=BATCH_N, shuffle=False, collate_fn=collate_fn
)

model.eval()
all_preds, all_golds = [], []

with torch.no_grad():
    for X, y, mask in test_loader:
        batch_preds = model.decode(X, mask)
        for pred_seq, gold_seq, m in zip(batch_preds, y, mask):
            valid_len = m.sum().item()
            all_preds.append([ner_labels[i] for i in pred_seq[:valid_len]])
            all_golds.append([ner_labels[i] for i in gold_seq[:valid_len].tolist()])

In [12]:
from seqeval.metrics import classification_report
from seqeval.metrics import f1_score

print(f"F1-score: {f1_score(all_golds, all_preds):.2f}")
print(classification_report(all_golds, all_preds))

F1-score: 0.78
              precision    recall  f1-score   support

         LOC       0.84      0.82      0.83      1668
        MISC       0.58      0.62      0.60       702
         ORG       0.70      0.72      0.71      1661
         PER       0.91      0.86      0.89      1617

   micro avg       0.79      0.77      0.78      5648
   macro avg       0.76      0.75      0.76      5648
weighted avg       0.79      0.77      0.78      5648

